In [1]:

from  google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!pip install  langchain sentence-transformers faiss-cpu pypdf transformers torch langchain-community langchain-huggingface

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
base_path = Path('/content/drive/MyDrive')
load= DirectoryLoader('/content/drive/MyDrive/classified_docs', glob='*.pdf',loader_cls=PyPDFLoader)
pag=load.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=50)
docs=text_splitter.split_documents(pag)
for idx,doc in enumerate(docs):
   print(f"[{idx}].{doc}\n\n")

[0].page_content='TRIBE: TRImodal Brain Encoder
for whole-brain fMRI response prediction
Stéphane d’Ascoli
Meta AI
sdascoli@meta.com
Jérémy Rapin
Meta AI
jrapin@meta.com
Yohann Benchetrit
Meta AI
ybenchetrit@meta.com
Hubert Banville
Meta AI
hubertjb@meta.com
Jean-Rémi King
Meta AI
jeanremi@meta.com
Abstract
Historically, neuroscience has progressed by fragmenting into specialized domains,' metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'author': "Stéphane d'Ascoli; Jérémy Rapin; Yohann Benchetrit; Hubert Banville; Jean-Rémi King", 'doi': 'https://doi.org/10.48550/arXiv.2507.22229', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'TRIBE: TRImodal Brain Encoder for whole-brain fMRI response prediction', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2507.22229v1', 'source': '/content/drive/MyDriv

In [5]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
embeddings=HuggingFaceEmbeddings(model_name='ibm-granite/granite-embedding-english-r2')
database=FAISS.from_documents(docs, embeddings)
database.save_local('faiss_index_hf')

/tmp/ipython-input-2793018450.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='ibm-granite/granite-embedding-english-r2')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or 

modules.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

W0907 08:44:36.400000 623 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


In [6]:

database = FAISS.load_local("faiss_index_hf", embeddings, allow_dangerous_deserialization=True)

query = "Brain based Artificial intelligence"
similar_docs = database.similarity_search(query, k=3)

for doc in similar_docs:
    print(doc.page_content[:300] + "...\n---")

Brain-inspired Artificial Intelligence: A Comprehensive Review • 5
Brain-inspired 
Artificial Intelligence
SECTION 2 AI Learning 
from Human Brain
Neural Architecture
Learning Mechanism
Attention and Focus
Memory and Recall
Consciousness
Creativity and 
Imagination
SECTION  3 
Methodologies
Physical...
---
Brain-inspired Artificial Intelligence: A Comprehensive Review • 25
brain functions can significantly contribute to creating intelligent machines. Effectively integrating neuroscience
concepts into AI requires a solid grasp of fundamental areas in neuroscience and cognitive science [177].
The two fi...
---
Brain-inspired Artificial Intelligence: A Comprehensive Review • 3
Table 1. Differences between brain-inspired AI and traditional AI
Aspect Brain-inspired AI Traditional AI
Learning Approach Mimics human brain learning (e.g., neural
networks)
Rule-based, predefined algorithms
Adaptability High, capa...
---


In [13]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, AutoModelForSeq2SeqLM
from langchain.chains import RetrievalQA
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
model_name = "google/flan-t5-large"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
memory=ConversationBufferMemory()
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=1024
)

llm = HuggingFacePipeline(pipeline=pipe)


qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    memory=memory,
    chain_type="stuff",
    retriever=database.as_retriever()

)
print("Starting chat convo if you wanna end one just type END")
while True:
    user = input("Yes you: ").strip()
    if user.strip().lower() == "end":
        break
    try:
        respo = qa_chain.invoke({"query": user})
        print(f"Asistent: {respo['result']}\n")
    except Exception as e:
        print(f"An error occurred: {e}")


Device set to use cuda:0


Starting chat convo if you wanna end one just type END
Yes you: What is TRIBE ?
Asistent: the first deep neural network trained to predict brain responses to stimuli across multiple modalities, cortical areas and individuals

Yes you: What TRIBE predicts ?
Asistent: brain responses to videos across diverse regions

Yes you: Who are the creators of TRIBE ?
Asistent: Stéphane d’Ascoli Meta AI sdascoli@meta.com Jérémy Rapin Meta AI jrapin@meta.com Yohann Benchetrit Meta AI ybenchetrit@meta.com Hubert Banville Meta AI hubertjb@meta.com Jean-Rémi King Meta AI jeanremi@meta.com

Yes you: What is BIAI inspiration ?
Asistent: BIAI refers to AI systems and algorithms that take inspiration from the biological structure, function, and principles of the human brain and neural system

Yes you: and what are DBNs ?
Asistent: generative models composed of stacked Restricted Boltzmann Machines (RBMs)

Yes you: how DBNs learn ?
Asistent: layer-by-layer through unsupervised techniques like contrastive di

#Podsumowanie zadania:
1)Siemanko ogolnie wybaczcie,że nie ma jakiś super podpisów przy danych partach notebooka, ale jestem na wakajkach i w sumie, chciałbym to skończyć szybciutko(Już wróciłem jak coś)  
2
)Przechodząc do samego Asystenta, jeśli ktoś z sprawdzających ma ochotę z chęcią podeśle, folder ogólna tematyka: badania mózgu, fMRI, i budowa modeli na bazie ludzkiego mózgu
3
)Strona techniczna, do embbedingu tych danych użyłem modelu od ibm,model ten generuje wektory o wielkości 768, czyli bodajże takie same jak ten użyty w notebooku, jest przeznaczony tylko do tektsu po angielsku, well nawet nazwa to potwierdza, model został wytrenowany na dużej ilości danych, co sprawia, ze jest precyzjny, o i ciekawostka potrafi wykrywać niuanse językowe czyli z Asystentem który go wykorzystuje można pogadać o Szekspirze.

4)Dalej stosowalem mniej więcej to co bylo w notatniku, aż do użycia samego modelu jezykowego, który ma być moim asystentem, więc pytanie czemu Flan large, a nie small Flan Large jest OGROMNY, 780 milionów parametrów, choć oczywiście zużywa więcej zasobów zarówno ramu jak i miejsca na dysku-ok 1GB, a small około 300MB,różnica jest jednak spora small ma tylko 80 mln parametrów, a że korzystam z serwerów Google to mogę je troche zajechać:))))

5)Ogólnie z przemyśleń na koniec chyba zadanie które mi najbardziej przypadło do gustu, pozdrawiam prowadzących

6)Czekając na przypisanie grup troche pogadalem z gagatkiem i pod koniec wyskoczylo mi ostrzeżenie o niskiej wydajności gpu, tem problem mozna usunąć dodając batche, Data_Set w pipeline
